In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
import os
from dotenv import load_dotenv
from tqdm import tqdm
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score

load_dotenv()
os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")
tqdm.pandas()

c:\Users\Tatha\anaconda3\envs\lead-routing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# loading cleaned data from data prep notebook
df = pd.read_csv('../data/cleaned/filtered_business_leads.csv')

In [3]:
# sorting unique labels so mapping is consistent across runs
sorted_labels = sorted(df['queue'].unique())
queue_mapping = {label: idx for idx, label in enumerate(sorted_labels)}

In [ ]:

# mapping string labels to integers
df['label'] = df['queue'].map(queue_mapping)

# saving mapping for inference later
with open(f'../models/routing_model/queue_mapping.pkl', 'wb') as f:
    pickle.dump(queue_mapping, f)
print("Label mapping:")
print(queue_mapping)
print(f"\nTotal classes: {len(queue_mapping)}")

Label mapping:
{'Billing': 0, 'Customer Service': 1, 'Sales': 2, 'Tech Support': 3}

Total classes: 4


In [7]:
df.head()

,type,queue,query,label
0,Incident,Tech Support,"Account Disruption Dear Customer Support Team,...",3
1,Request,Customer Service,Query About Smart Home System Integration Feat...,1
2,Request,Billing,Inquiry Regarding Invoice Details Dear Custome...,0
3,Problem,Sales,Question About Marketing Agency Software Compa...,2
4,Request,Tech Support,"Feature Query Dear Customer Support,\n\nI hope...",3


In [ ]:
# injecting short query examples per merged class
short_queries = [
    # Tech Support
    {"query": "My laptop is not turning on", "queue": "Tech Support"},
    {"query": "The software keeps crashing", "queue": "Tech Support"},
    {"query": "I cannot connect to WiFi", "queue": "Tech Support"},
    {"query": "The website is down", "queue": "Tech Support"},
    {"query": "Is there scheduled maintenance today?", "queue": "Tech Support"},
    {"query": "Your service is not working", "queue": "Tech Support"},
    {"query": "I need IT help with my computer", "queue": "Tech Support"},
    {"query": "My screen is not displaying properly", "queue": "Tech Support"},

    # Customer Service
    {"query": "I want to return my laptop", "queue": "Customer Service"},
    {"query": "This laptop needs to be changed under warranty", "queue": "Customer Service"},
    {"query": "How do I exchange a defective product?", "queue": "Customer Service"},
    {"query": "I need a replacement for my broken device", "queue": "Customer Service"},
    {"query": "I need help with my account", "queue": "Customer Service"},
    {"query": "Can you explain how this works?", "queue": "Customer Service"},
    {"query": "I have a question about my order", "queue": "Customer Service"},

    # Billing
    {"query": "I was charged twice", "queue": "Billing"},
    {"query": "My invoice has wrong charges", "queue": "Billing"},
    {"query": "When is my next payment due?", "queue": "Billing"},
    {"query": "Can you refund my payment?", "queue": "Billing"},
    {"query": "I have a general question about your services", "queue": "Billing"},
    {"query": "I need help understanding my bill", "queue": "Billing"},

    # Sales
    {"query": "I want to buy a laptop", "queue": "Sales"},
    {"query": "What are your pricing plans?", "queue": "Sales"},
    {"query": "Do you offer student discounts?", "queue": "Sales"},
    {"query": "I am interested in purchasing your software", "queue": "Sales"},
    {"query": "Can I get a demo of your product?", "queue": "Sales"},
    {"query": "How much does your enterprise plan cost?", "queue": "Sales"},
    {"query": "What products do you sell?", "queue": "Sales"},
]
short_df = pd.DataFrame(short_queries)
short_df['label'] = short_df['queue'].map(queue_mapping)

Dataset size after injection: 24573


In [ ]:
#splitting the short text data for train test
from sklearn.model_selection import train_test_split as sk_split
short_train, short_test = sk_split(
    short_df, test_size=0.2, random_state=42, stratify=short_df['queue']
)

In [ ]:
# duplicating training short data to give weight against large email dataset
short_train = pd.concat([short_train] * 15, ignore_index=True)

print(f"Short query train: {len(short_train)}")
print(f"Short query test:  {len(short_test)}")

In [ ]:
#splitting the long text data for train test

email_df = df.copy()  # this is original long text df before short query injection
email_train, email_test = sk_split(
    email_df, test_size=0.2, random_state=42, stratify=email_df['queue']
)

In [ ]:
# combining train portions and test portions separately to prevent data leakage
train_df = pd.concat([email_train, short_train], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.concat([email_test, short_test], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

for df in [train_df, test_df]:
    df.dropna(subset=['label'], inplace=True)
    df['label'] = df['label'].astype(int)

In [ ]:
print(f"Final train Dataset size after injection:: {len(train_df)}")
print(f"Final test Dataset size after injection::  {len(test_df)}")

In [ ]:
print(f"train Dataset labels::{train_df['label'].value_counts()}")
print(f"test Dataset labels::{test_df['label'].value_counts()}")

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24573 entries, 0 to 24572
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   type    24153 non-null  str  
 1   queue   24573 non-null  str  
 2   query   24573 non-null  str  
 3   label   24573 non-null  int64
dtypes: int64(1), str(3)
memory usage: 10.7 MB


In [ ]:
# converting to HuggingFace dataset format
hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[['query', 'label']], preserve_index=False),
    "test":  Dataset.from_pandas(test_df[['query', 'label']], preserve_index=False)
})


print(f"Train size: {len(hf_dataset['train'])}")
print(f"Test size:  {len(hf_dataset['test'])}")

Train size: 19658
Test size:  4915
label
3    11089
1     9719
0     2968
2      797
Name: count, dtype: int64


In [ ]:
# loading distilbert tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# tokenizing with fixed 128 token length
def tokenize_fn(examples):
    return tokenizer(examples["query"], padding="max_length", truncation=True, max_length=128)

# applying tokenizer to full dataset
tokenized_datasets = hf_dataset.map(tokenize_fn, batched=True)
print("Tokenizing done")
print(tokenized_datasets)

In [ ]:
# loading distilbert with correct number of output heads
num_labels = len(queue_mapping)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="single_label_classification"
)

print(f"Model loaded with {num_labels} output classes")

In [ ]:
# computing accuracy and weighted f1 after each eval
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

In [ ]:
# setting up training hyperparameters
training_args = TrainingArguments(
    output_dir="routing_model_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,              # bumped from 3 as distilbert needs more passes on multi-class
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy", # tracking accuracy
    greater_is_better=True,
    report_to="none"
)

In [ ]:
# setting up HF trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# running training loop
trainer.train()

In [ ]:
# saving model and tokenizer together so inference loads from same place
MODEL_SAVE_PATH = "../models/routing_model/distilbert-ticket-routing"
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)  # important — original code was missing this

print(f"Model and tokenizer saved to {MODEL_SAVE_PATH}")

Manual test

In [ ]:
# loading saved model and tokenizer
MODEL_SAVE_PATH = "../models/routing_model/distilbert-ticket-routing"
tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)

# loading the label mapping and inverting it for prediction → name lookup
with open(f'../models/routing_model/queue_mapping.pkl', 'rb') as f:
    queue_mapping =pickle.load(f)
id_to_queue = {v: k for k, v in queue_mapping.items()}

# setting eval mode once before inference
model.eval()

print("Model ready for inference")
print(f"Routing to {len(id_to_queue)} queues: {list(id_to_queue.values())}")

In [ ]:
# predicting department for a single query string
def predict_queue(text, model, tokenizer, id_to_queue):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    pred_idx = torch.argmax(outputs.logits, dim=-1).item()
    confidence = torch.nn.functional.softmax(outputs.logits, dim=-1)[0][pred_idx].item()
    return id_to_queue[pred_idx], confidence

In [ ]:
# running live terminal classifier
def interactive_routing_classifier(model, tokenizer, id_to_queue):
    print("\n" + "="*50)
    print("      LIVE TICKET ROUTING CLASSIFIER")
    print("  Type your support query. Type 'quit' to exit.")
    print("="*50 + "\n")

    while True:
        user_input = input("Enter support query: ").strip()

        if user_input.lower() == 'quit':
            print("\nExiting. Bye!")
            break

        if not user_input:
            print("Input cannot be empty. Try again.\n")
            continue

        queue, confidence = predict_queue(user_input, model, tokenizer, id_to_queue)

        print("-" * 40)
        print(f"✅ Routed to: [{queue}]")
        print(f"📊 Confidence: {confidence:.2%}")
        print("-" * 40 + "\n")

interactive_routing_classifier(model, tokenizer, id_to_queue)